# Analyze New Dataset
Run the cell below to initialize the Spark Session and load the `dataset.csv` into memory as a temporary view.

In [1]:
import os
import sys

venv_site_packages = os.path.join(os.getcwd(), 'venv', 'Lib', 'site-packages')
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

current_dir = os.getcwd()
os.environ['HADOOP_HOME'] = os.path.join(current_dir, 'hadoop')
os.environ['JAVA_HOME'] = os.path.join(current_dir, 'jdk-17')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AnalyzeDataset") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("dataset.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("my_table")

print("✅ Spark is Ready! Temp Table available: 'my_table'")
df.printSchema()

✅ Spark is Ready! Temp Table available: 'my_table'
root
 |-- ID : integer (nullable = true)
 |--  Name: string (nullable = true)
 |-- Amount($): string (nullable = true)
 |-- is Active: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Updated At: string (nullable = true)



In [ ]:
query = """
SELECT * 
FROM my_table
LIMIT 50
"""

spark.sql(query).show()

+---+-------+---------+---------+---------------+-------+---------------+
|ID |   Name|Amount($)|is Active|     Order Date|    Sex|     Updated At|
+---+-------+---------+---------+---------------+-------+---------------+
|152|   Ånna|    1 200| Inactive|2/26/2023 22:12| FEMALE| 6/1/2023 16:02|
|415|  Renée|10.000,00|        1|1/15/2023 10:57|      f|2/13/2023 19:50|
|119|   Judy| 1,000.50|   Active| 6/2/2023 17:14|      M| 4/6/2023 15:32|
|365|   Ivan|    -5000|        N| 2/2/2023 21:16|      f|2/15/2023 21:07|
| 30|    Bob|       NA|     TRUE|           NULL|  Other| 1/13/2023 1:20|
|403|  Frank|      500|   Active| 3/31/2023 0:58|      f|  4/6/2023 5:51|
| 64|  David|     2000|       NO|1/16/2023 21:43|      f|  6/4/2023 1:03|
|482|  Alice| 2.000,50|        Y|6/10/2023 16:42| FEMALE|6/10/2023 20:17|
|442| Hannah|10.000,00| Inactive| 2/8/2023 14:43|      F|  2/4/2023 7:50|
|448|  Renée| 2.000,50|   Active| 5/1/2023 17:50|      f| 1/22/2023 2:21|
| 31|    Bob| 2.000,50|       NO| 3/21

In [6]:
import re

# Loop through and clean all column names dynamically
for col_name in df.columns:
    # 1. Strip leading/trailing spaces & convert to lowercase
    clean_name = col_name.strip().lower()
    # 2. Replace non-alphanumeric characters (spaces, $, (), etc.) with underscores
    clean_name = re.sub(r'[^a-z0-9]+', '_', clean_name)
    # 3. Trim extra leading/trailing underscores
    clean_name = clean_name.strip('_')
    
    # Rename in DataFrame
    df = df.withColumnRenamed(col_name, clean_name)

# Re-register the temp view with the standardized column names
df.createOrReplaceTempView("my_table")

# Print the updated schema
print("✅ Standardized Schema:")
df.printSchema()


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [7]:
query = """
SELECT * 
FROM my_table
LIMIT 50
"""

spark.sql(query).show()

+---+-------+---------+---------+---------------+-------+---------------+
| id|   name|   amount|is_active|     order_date|    sex|     updated_at|
+---+-------+---------+---------+---------------+-------+---------------+
|152|   Ånna|    1 200| Inactive|2/26/2023 22:12| FEMALE| 6/1/2023 16:02|
|415|  Renée|10.000,00|        1|1/15/2023 10:57|      f|2/13/2023 19:50|
|119|   Judy| 1,000.50|   Active| 6/2/2023 17:14|      M| 4/6/2023 15:32|
|365|   Ivan|    -5000|        N| 2/2/2023 21:16|      f|2/15/2023 21:07|
| 30|    Bob|       NA|     TRUE|           NULL|  Other| 1/13/2023 1:20|
|403|  Frank|      500|   Active| 3/31/2023 0:58|      f|  4/6/2023 5:51|
| 64|  David|     2000|       NO|1/16/2023 21:43|      f|  6/4/2023 1:03|
|482|  Alice| 2.000,50|        Y|6/10/2023 16:42| FEMALE|6/10/2023 20:17|
|442| Hannah|10.000,00| Inactive| 2/8/2023 14:43|      F|  2/4/2023 7:50|
|448|  Renée| 2.000,50|   Active| 5/1/2023 17:50|      f| 1/22/2023 2:21|
| 31|    Bob| 2.000,50|       NO| 3/21

In [8]:
from pyspark.sql.functions import col, trim, when, lower, regexp_replace, expr

# Step 1: Standardize Column Names using Regex
import re
for c in df.columns:
    clean_c = re.sub(r'[^a-z0-9]', '_', c.strip().lower())
    clean_c = re.sub(r'_+', '_', clean_c).strip('_')
    df = df.withColumnRenamed(c, clean_c)

# -------------------------------------------------------------
# Step 2: Convert Placeholder Strings ('NA', '-', 'unknown', '') to NULL
# -------------------------------------------------------------
string_cols = [item[0] for item in df.dtypes if item[1] == 'string']

for c in string_cols:
    df = df.withColumn(
        c,
        when(trim(col(c)).isin("NA", "-", "unknown", ""), None)
        .otherwise(trim(col(c)))
    )

# -------------------------------------------------------------
# Step 3: Clean & Normalize String Categories
# -------------------------------------------------------------

# Normalize 'sex' column (M / Male -> Male, F / Female -> Female, O / Other -> Other)
df = df.withColumn(
    "sex",
    when(lower(col("sex")).isin("m", "male"), "Male")
    .when(lower(col("sex")).isin("f", "female"), "Female")
    .when(lower(col("sex")).isin("o", "other"), "Other")
    .otherwise("Unknown")
)

# Normalize 'is_active' column to True / False
df = df.withColumn(
    "is_active",
    when(lower(col("is_active")).isin("1", "active", "y", "yes", "true"), True)
    .when(lower(col("is_active")).isin("0", "inactive", "n", "no", "false"), False)
    .otherwise(None)
)

# -------------------------------------------------------------
# Step 4: Clean Dirty Amount Strings into Double/Numeric
# (Handles spaces, '10.000,00' European formats, and '1,000.50')
# -------------------------------------------------------------
df = df.withColumn(
    "amount_clean",
    regexp_replace(col("amount"), r"\s+", "")  # remove internal spaces
)
df = df.withColumn(
    "amount_clean",
    when(col("amount_clean").rlike(r"\d+\.\d+,\d+"), 
         regexp_replace(regexp_replace(col("amount_clean"), r"\.", ""), r",", "."))
    .when(col("amount_clean").rlike(r"\d+,\d+\.\d+"), 
         regexp_replace(col("amount_clean"), r",", ""))
    .otherwise(col("amount_clean")).cast("double")
)

# -------------------------------------------------------------
# Step 5: Handle Missing Values (Nulls)
# -------------------------------------------------------------

# Option A: Drop rows where critical columns (id or order_date) are NULL
df_cleaned = df.dropna(subset=["id", "order_date"])

# Option B: Fill remaining missing values with defaults
df_cleaned = df_cleaned.fillna({
    "name": "Anonymous",
    "sex": "Unknown",
    "amount_clean": 0.0
})

# Re-register view
df_cleaned.createOrReplaceTempView("clean_dataset")

print("✅ Data Cleaning Complete!")
df_cleaned.show(10)


✅ Data Cleaning Complete!
+---+------+---------+---------+---------------+------+---------------+------------+
| id|  name|   amount|is_active|     order_date|   sex|     updated_at|amount_clean|
+---+------+---------+---------+---------------+------+---------------+------------+
|152|  Ånna|    1 200|    false|2/26/2023 22:12|Female| 6/1/2023 16:02|      1200.0|
|415| Renée|10.000,00|     true|1/15/2023 10:57|Female|2/13/2023 19:50|     10000.0|
|119|  Judy| 1,000.50|     true| 6/2/2023 17:14|  Male| 4/6/2023 15:32|      1000.5|
|365|  Ivan|    -5000|    false| 2/2/2023 21:16|Female|2/15/2023 21:07|     -5000.0|
|403| Frank|      500|     true| 3/31/2023 0:58|Female|  4/6/2023 5:51|       500.0|
| 64| David|     2000|    false|1/16/2023 21:43|Female|  6/4/2023 1:03|      2000.0|
|482| Alice| 2.000,50|     true|6/10/2023 16:42|Female|6/10/2023 20:17|      2000.5|
|442|Hannah|10.000,00|    false| 2/8/2023 14:43|Female|  2/4/2023 7:50|     10000.0|
|448| Renée| 2.000,50|     true| 5/1/20

In [9]:
from pyspark.sql.functions import concat, lit, format_number

# Format the clean numeric amount as currency string: $10,000.00
df = df.withColumn(
    "amount_formatted",
    concat(lit("$"), format_number(col("amount_clean"), 2))
)

df.select("name", "amount", "amount_clean", "amount_formatted").show(10)


+------+---------+------------+----------------+
|  name|   amount|amount_clean|amount_formatted|
+------+---------+------------+----------------+
|  Ånna|    1 200|      1200.0|       $1,200.00|
| Renée|10.000,00|     10000.0|      $10,000.00|
|  Judy| 1,000.50|      1000.5|       $1,000.50|
|  Ivan|    -5000|     -5000.0|      $-5,000.00|
|   Bob|     NULL|        NULL|            NULL|
| Frank|      500|       500.0|         $500.00|
| David|     2000|      2000.0|       $2,000.00|
| Alice| 2.000,50|      2000.5|       $2,000.50|
|Hannah|10.000,00|     10000.0|      $10,000.00|
| Renée| 2.000,50|      2000.5|       $2,000.50|
+------+---------+------------+----------------+
only showing top 10 rows

